In [1]:
from BEATs import BEATs, BEATsConfig
import torch

In [2]:
model_path = '/home/user/hyunjun/pretrained_weights/BEATs_iter3_plus_AS2M.pt'
checkpoint = torch.load(model_path)
cfg = BEATsConfig(checkpoint['cfg'])
model = BEATs(cfg).float()
model.load_state_dict(checkpoint['model'])

/home/user/anaconda3/envs/HJ_audio/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


<All keys matched successfully>

In [3]:
import numpy as np

def pad_or_truncate_to_seconds(wav, sr, target_seconds=10):
    target_length = target_seconds * sr  # 목표 길이 (샘플 수)
    current_length = wav.shape[0]  # 현재 오디오 길이
    
    padding_mask = np.zeros(target_length)
    
    if current_length < target_length:
        pad_size = target_length - current_length
        wav = np.pad(wav, (0, pad_size), 'constant')
        padding_mask[current_length:] = 1
    elif current_length > target_length:
        max_start = current_length - target_length
        start_idx = np.random.randint(0, max_start + 1)
        wav = wav[start_idx:start_idx + target_length]
    else:
        # 길이가 정확히 맞는 경우, padding_mask는 모두 False
        pass
    
    return wav, padding_mask

In [4]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
df = pd.read_csv('../../../../data/dev_train.csv')
dummy = df.loc[0, 'audio_path']
dummy

'/mnt/storage1/asd/2025/data/ToyCar/train/section_00_source_train_normal_0000_car_D2_spd_40V_mic_1.wav'

In [5]:
wav, sr = sf.read(dummy)

wav = wav - wav.mean()  # Zero-mean normalization
wav, padding_mask = pad_or_truncate_to_seconds(wav, sr, target_seconds=10)

wav = torch.from_numpy(wav).float()
wav = wav.unsqueeze(dim=0)
padding_mask = torch.from_numpy(padding_mask).float()
padding_mask = padding_mask.unsqueeze(dim=0)

In [6]:
def hook_fn(module, input, output, layer_outputs):
    # BEATs: 단일 텐서 출력, EAT: (x, t) 튜플 출력
    if isinstance(output, tuple):  # EAT의 AltBlock
        x = output[0].detach()  # 메인 출력 선택
        if x.shape[1] == 513:  # 클래스 토큰 제거
            x = x[:, 1:, :]  # (batch, 512, 768)
    else:  # BEATs의 TransformerSentenceEncoderLayer
        x = output.detach()  # (batch, 512, 768)
    layer_outputs.append(x)

In [7]:
layer_outputs = []
hooks = []
for layer in model.encoder.layers:
    hook = layer.register_forward_hook(lambda module, input, output: hook_fn(module, input, output, layer_outputs))
    hooks.append(hook)

x, _ = model.extract_features(source=wav, padding_mask=padding_mask)

for hook in hooks:
    hook.remove()

concatenated = torch.cat(layer_outputs, dim=-1)  # (batch, 512, 12*768)

In [8]:
len(layer_outputs)

12

In [18]:
layer_outputs

[tensor([[[-0.1434, -0.2523,  0.2647,  ..., -0.1921,  0.0685,  0.1213]],
 
         [[-0.0218,  0.2048,  0.1930,  ..., -0.0278,  0.2315,  0.0826]],
 
         [[-0.0601,  0.0718,  0.1394,  ...,  0.0159,  0.2036, -0.0131]],
 
         ...,
 
         [[-0.0185, -0.0695,  0.1851,  ..., -0.0144,  0.2726, -0.1556]],
 
         [[ 0.0932, -0.0936,  0.1950,  ...,  0.1309,  0.2349, -0.2657]],
 
         [[-0.0421,  0.4359, -0.9109,  ...,  0.1521,  0.1905, -0.0784]]]),
 tensor([[[-0.3551, -0.2235,  0.3780,  ...,  0.0063,  0.0670,  0.5194]],
 
         [[ 0.0093,  0.1979,  0.1057,  ...,  0.1677,  0.2126,  0.2047]],
 
         [[ 0.1005, -0.0515,  0.2942,  ...,  0.2427, -0.0779,  0.0284]],
 
         ...,
 
         [[-0.0450,  0.0466,  0.1378,  ..., -0.0626,  0.0255,  0.0796]],
 
         [[ 0.0890, -0.0104,  0.2466,  ...,  0.3314, -0.0807, -0.2452]],
 
         [[ 0.0376,  0.3353, -0.4737,  ...,  0.3665,  0.0201, -0.0185]]]),
 tensor([[[-0.1397, -0.3105,  0.1541,  ...,  0.2513,  0.2071,  0.445

In [16]:
layer_outputs[0].shape

torch.Size([496, 1, 768])

In [20]:
torch.cat(layer_outputs, dim=-1).transpose(0, 1).shape

torch.Size([1, 496, 9216])